# AgroPest-12: Dataset Exploration & Preprocessing

**Student:** Md. Sadik Shahriar  
**Dataset:** AgroPest-12 &mdash; Image Dataset for Crop Pest Detection  
**Source:** [Kaggle &mdash; Rupankar Majumdar](https://www.kaggle.com/datasets/rupankarmajumdar/crop-pests-dataset)  
**Course:** CSE 445 &mdash; Computer Vision | Assignment 1  

This notebook performs **Exploratory Data Analysis (EDA)** on the AgroPest-12 crop pest dataset.  
The dataset contains **13,143 images** across **12 pest classes** with YOLO-format bounding box annotations,  
pre-split into train (11,502), validation (1,095), and test (546) sets.

In [ ]:
# ============================================================
# CONFIG — All paths, class names, and key parameters
# ============================================================
import os

# Kaggle dataset paths
DATASET_PATH = "/kaggle/input/datasets/rupankarmajumdar/crop-pests-dataset"
TRAIN_IMG = os.path.join(DATASET_PATH, "train", "images")
TRAIN_LBL = os.path.join(DATASET_PATH, "train", "labels")
VALID_IMG = os.path.join(DATASET_PATH, "valid", "images")
VALID_LBL = os.path.join(DATASET_PATH, "valid", "labels")
TEST_IMG  = os.path.join(DATASET_PATH, "test", "images")
TEST_LBL  = os.path.join(DATASET_PATH, "test", "labels")

# Class names (YOLO format: 0-indexed)
CLASS_NAMES = {
    0: "Ants", 1: "Bees", 2: "Beetles", 3: "Caterpillars",
    4: "Earthworms", 5: "Earwigs", 6: "Grasshoppers", 7: "Moths",
    8: "Slugs", 9: "Snails", 10: "Wasps", 11: "Weevils"
}
NUM_CLASSES = 12

# Image parameters
IMAGE_SIZE = 640  # dataset native resolution

# COCO small / medium / large thresholds (area in pixels²)
COCO_SMALL  = 32**2      # < 1024 px²
COCO_MEDIUM = 96**2      # 1024–9216 px²

# Output directory for saved artifacts
OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("CONFIG loaded — 12 classes | 640×640 images | pre-split dataset")

In [ ]:
# ============================================================
# Imports & Plot Styling
# ============================================================
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
from pathlib import Path
from collections import Counter, defaultdict
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams.update({
    'figure.dpi': 100,
    'savefig.dpi': 100,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
sns.set_style("whitegrid")

# Distinct colors for 12 classes
CLASS_COLORS = [
    "#e6194b", "#3cb44b", "#ffe119", "#4363d8",
    "#f58231", "#911eb4", "#46f0f0", "#f032e6",
    "#bcf60c", "#fabebe", "#008080", "#e6beff"
]

print("Libraries imported.")

## 1. Dataset Overview

AgroPest-12 is a curated dataset of crop pest images captured in natural agricultural environments.
Each image has corresponding YOLO-format bounding box annotations (`<class_id> <cx> <cy> <w> <h>` with
normalised coordinates). The dataset arrives **pre-split** into training, validation, and test
directories. The following cell verifies file counts, label–image pairing, and checks for corrupt
files before proceeding to EDA.

In [ ]:
# ============================================================
# Task 5.1 — Dataset Integrity Check
# ============================================================
def check_split(img_dir, lbl_dir, split_name, spot_check=200):
    """Count images/labels, detect orphans, spot-check for corrupt JPEGs."""
    imgs = sorted([f for f in os.listdir(img_dir)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    lbls = sorted([f for f in os.listdir(lbl_dir)
                   if f.lower().endswith('.txt')])

    img_stems = {os.path.splitext(f)[0] for f in imgs}
    lbl_stems = {os.path.splitext(f)[0] for f in lbls}

    orphan_imgs = img_stems - lbl_stems  # image with no label
    orphan_lbls = lbl_stems - img_stems  # label with no image

    corrupt = 0
    for f in imgs[:spot_check]:
        img = cv2.imread(os.path.join(img_dir, f))
        if img is None:
            corrupt += 1

    print(f"{split_name:>6}: {len(imgs):>6} imgs | {len(lbls):>6} lbls | "
          f"orphan I={len(orphan_imgs)} L={len(orphan_lbls)} | corrupt≈{corrupt}")

    return {
        "split": split_name,
        "images": len(imgs),
        "labels": len(lbls),
        "orphan_images": len(orphan_imgs),
        "orphan_labels": len(orphan_lbls),
        "corrupt_spotted": corrupt,
    }

results = []
for path_tuple, name in [
    ((TRAIN_IMG, TRAIN_LBL), "train"),
    ((VALID_IMG, VALID_LBL), "valid"),
    ((TEST_IMG,  TEST_LBL),  "test"),
]:
    results.append(check_split(*path_tuple, name))

df = pd.DataFrame(results).set_index("split")
total_imgs = df["images"].sum()
total_lbls = df["labels"].sum()
print(f"\n│ Total: {total_imgs} images, {total_lbls} labels │")
display(df)

## 2. Class Distribution

Parsing every label file across the **training set** to count per-class bounding-box instances.
A highly imbalanced dataset would influence augmentation choices (e.g., enabling mosaic or
mixup to up-sample rare classes) and the choice of loss function (focal vs. standard).

In [ ]:
# ============================================================
# Task 5.1 — Class Distribution (instance count per class)
# ============================================================

def count_instances(lbl_dir):
    """Count per-class instance totals from all .txt label files."""
    counter = Counter()
    lbl_files = [f for f in os.listdir(lbl_dir) if f.endswith('.txt')]
    for fname in tqdm(lbl_files, desc="Parsing labels", leave=False):
        with open(os.path.join(lbl_dir, fname), 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(float(parts[0]))
                    counter[cls_id] += 1
    return counter

# Count instances in training set
train_counts = count_instances(TRAIN_LBL)

# Build DataFrame
rows = []
for cls_id in sorted(CLASS_NAMES.keys()):
    rows.append({
        "Class": CLASS_NAMES[cls_id],
        "Instances": train_counts.get(cls_id, 0),
    })
df_cls = pd.DataFrame(rows).set_index("Class")

# Bar chart
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(df_cls.index, df_cls["Instances"],
              color=[CLASS_COLORS[i] for i in range(NUM_CLASSES)], edgecolor='black')

ax.set_title("Class Distribution — Training Set Instances")
ax.set_ylabel("Number of instances")
ax.tick_params(axis='x', rotation=45)

# Annotate bars
for bar, val in zip(bars, df_cls["Instances"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
            str(val), ha='center', va='bottom', fontsize=9, fontweight='bold')

# Imbalance note
max_val = df_cls["Instances"].max()
min_val = df_cls["Instances"].min()
imbalance_ratio = max_val / min_val if min_val > 0 else float('inf')
ax.text(0.99, 0.95, f"Imbalance ratio (max/min): {imbalance_ratio:.1f}",
        transform=ax.transAxes, ha='right', va='top',
        fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "class_distribution.png"))
plt.show()

display(df_cls)

## 3. Image Properties

Spot-checking image resolution, aspect ratio, and file format across the dataset.  
The dataset documentation states all images are **640×640 JPEG**, but we verify this
empirically. Any deviation from square would require rectangular training (`rect=True`).

In [ ]:
# ============================================================
# Task 5.1 — Image Properties (resolution, aspect ratio, format)
# ============================================================

# Sample across all three splits for a representative view
sample_paths = []
for img_dir in [TRAIN_IMG, VALID_IMG, TEST_IMG]:
    imgs = sorted([f for f in os.listdir(img_dir)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    sample_paths += [os.path.join(img_dir, f) for f in imgs[:500]]  # 500 per split

resolutions = []
aspects = []
formats = set()
for p in tqdm(sample_paths, desc="Scanning images"):
    img = cv2.imread(p)
    if img is not None:
        h, w = img.shape[:2]
        resolutions.append((w, h))
        aspects.append(w / h)
        ext = os.path.splitext(p)[1].lower()
        formats.add(ext)

res_arr = np.array(resolutions)
asp_arr = np.array(aspects)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Width distribution
axes[0].hist(res_arr[:, 0], bins=20, color='steelblue', edgecolor='black')
axes[0].set_title("Image Width Distribution (all = 640)")
axes[0].set_xlabel("Width (px)")
axes[0].set_ylabel("Count")

# Aspect ratio
axes[1].hist(asp_arr, bins=20, color='darkorange', edgecolor='black')
axes[1].set_title("Aspect Ratio Distribution (w/h)")
axes[1].set_xlabel("Aspect Ratio")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "image_properties.png"))
plt.show()

print(f"File formats found: {formats}")
print(f"Unique resolutions: {set(tuple(r) for r in resolutions)}")
print(f"Aspect ratio range: [{asp_arr.min():.3f}, {asp_arr.max():.3f}]")

## 4. Bounding Box Statistics

Analysing box count per image, width / height / area distributions, and the proportion of
small vs. medium vs. large objects (using COCO thresholds: **small < 32²**, **medium 32²–96²**,
**large > 96²**). This directly influences model size selection and input resolution: a
dataset dominated by small objects benefits from higher resolution or a larger model.

In [ ]:
# ============================================================
# Task 5.1 — Bounding Box Statistics
# ============================================================

def parse_all_boxes(lbl_dir):
    """Parse every .txt label file and return arrays of [w, h, area, cls_id]."""
    all_w, all_h, all_area, all_cls = [], [], [], []
    boxes_per_img = []
    lbl_files = [f for f in os.listdir(lbl_dir) if f.endswith('.txt')]
    for fname in tqdm(lbl_files, desc="Parsing boxes", leave=False):
        count = 0
        with open(os.path.join(lbl_dir, fname), 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(float(parts[0]))
                    cx, cy, bw, bh = map(float, parts[1:5])
                    # Normalised -> absolute pixels (640 px)
                    w_abs = bw * IMAGE_SIZE
                    h_abs = bh * IMAGE_SIZE
                    area = w_abs * h_abs
                    all_w.append(w_abs)
                    all_h.append(h_abs)
                    all_area.append(area)
                    all_cls.append(cls_id)
                    count += 1
        boxes_per_img.append(count)
    return (np.array(all_w), np.array(all_h), np.array(all_area),
            np.array(all_cls), np.array(boxes_per_img))

# Parse training set
w_arr, h_arr, area_arr, cls_arr, boxes_per_img = parse_all_boxes(TRAIN_LBL)

# COCO size classification (absolute px on 640×640)
small_mask  = area_arr < COCO_SMALL
medium_mask = (area_arr >= COCO_SMALL) & (area_arr <= COCO_MEDIUM)
large_mask  = area_arr > COCO_MEDIUM

n_small  = small_mask.sum()
n_medium = medium_mask.sum()
n_large  = large_mask.sum()
total_boxes = len(area_arr)

# --- Subplot layout ---
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# A) Boxes-per-image histogram
axes[0, 0].hist(boxes_per_img, bins=30, color='steelblue', edgecolor='black')
axes[0, 0].set_title(f"Boxes per Image\nMedian={np.median(boxes_per_img):.0f}, Max={boxes_per_img.max()}")
axes[0, 0].set_xlabel("Number of boxes")
axes[0, 0].set_ylabel("Image count")

# B) Box width distribution
axes[0, 1].hist(w_arr, bins=50, color='darkorange', edgecolor='black')
axes[0, 1].set_title(f"Box Width (px)\nMean={w_arr.mean():.1f}, σ={w_arr.std():.1f}")
axes[0, 1].set_xlabel("Width (px)")

# C) Box height distribution
axes[0, 2].hist(h_arr, bins=50, color='forestgreen', edgecolor='black')
axes[0, 2].set_title(f"Box Height (px)\nMean={h_arr.mean():.1f}, σ={h_arr.std():.1f}")
axes[0, 2].set_xlabel("Height (px)")

# D) Box area distribution (log scale)
axes[1, 0].hist(area_arr, bins=60, color='mediumpurple', edgecolor='black')
axes[1, 0].set_yscale('log')
axes[1, 0].set_title(f"Box Area (px²)\nMedian={np.median(area_arr):.0f}")
axes[1, 0].set_xlabel("Area (px²)")

# E) COCO size pie
sizes = [n_small, n_medium, n_large]
labels_pie = [f"Small\n(< 32²)\n{n_small}",
              f"Medium\n(32²–96²)\n{n_medium}",
              f"Large\n(> 96²)\n{n_large}"]
colors_pie = ["#e74c3c", "#f39c12", "#27ae60"]
axes[1, 1].pie(sizes, labels=labels_pie, colors=colors_pie,
              autopct='%1.1f%%', startangle=140, textprops={'fontsize': 9})
axes[1, 1].set_title("COCO Size Classification")

# F) Boxes per class (training set)
per_cls_boxes = {cid: (cls_arr == cid).sum() for cid in range(NUM_CLASSES)}
cls_names = [CLASS_NAMES[i] for i in range(NUM_CLASSES)]
cls_counts = [per_cls_boxes[i] for i in range(NUM_CLASSES)]
axes[1, 2].bar(cls_names, cls_counts, color=[CLASS_COLORS[i] for i in range(NUM_CLASSES)], edgecolor='black')
axes[1, 2].set_title("Boxes per Class")
axes[1, 2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "bbox_statistics.png"))
plt.show()

# Summary table
print(f"Total boxes parsed: {total_boxes}")
print(f"Small  (< {COCO_SMALL:>5} px²): {n_small:>6}  ({100*n_small/total_boxes:.1f}%)")
print(f"Medium ( {COCO_SMALL}–{COCO_MEDIUM} px²): {n_medium:>6}  ({100*n_medium/total_boxes:.1f}%)")
print(f"Large  (> {COCO_MEDIUM:>5} px²): {n_large:>6}  ({100*n_large/total_boxes:.1f}%)")
print(f"Boxes-per-image: median={np.median(boxes_per_img):.1f}, mean={boxes_per_img.mean():.2f}, max={boxes_per_img.max()}")

## 5. Annotated Sample Images

Visualising ground-truth bounding boxes overlaid on sample images from the training set.
Boxes are **colour-coded by class** to give a qualitative sense of object size, density,
and class co-occurrence within a single image.

In [ ]:
# ============================================================
# Task 5.1 — Annotated Sample Visualisation
# ============================================================

def draw_boxes(img_path, lbl_path, ax):
    """Load image, parse labels, draw colour-coded boxes."""
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    ax.imshow(img)
    with open(lbl_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls_id = int(float(parts[0]))
            cx, cy, bw, bh = map(float, parts[1:5])
            x1 = int((cx - bw / 2) * w)
            y1 = int((cy - bh / 2) * h)
            box_w = int(bw * w)
            box_h = int(bh * h)

            color = CLASS_COLORS[cls_id]
            rect = mpatches.Rectangle((x1, y1), box_w, box_h,
                                      fill=False, edgecolor=color, linewidth=1.5)
            ax.add_patch(rect)
            ax.text(x1, max(y1 - 2, 6), CLASS_NAMES[cls_id],
                    fontsize=6, color='white', weight='bold',
                    bbox=dict(boxstyle='round,pad=0.1', facecolor=color, alpha=0.75))

    ax.axis('off')

# Gather diverse samples: up to 2 images per class
imgs_available = sorted(os.listdir(TRAIN_IMG))
collected = {i: [] for i in range(NUM_CLASSES)}
for fname in imgs_available:
    if all(len(v) >= 2 for v in collected.values()):
        break
    lbl_fname = os.path.splitext(fname)[0] + '.txt'
    lbl_path = os.path.join(TRAIN_LBL, lbl_fname)
    if not os.path.exists(lbl_path):
        continue
    # Determine which classes are in this image
    cls_in_img = set()
    with open(lbl_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                cls_in_img.add(int(float(parts[0])))
    for cid in cls_in_img:
        if len(collected[cid]) < 2:
            collected[cid].append((os.path.join(TRAIN_IMG, fname), lbl_path))
            break  # one image can cover multiple classes, but don't reuse too much

# Build flat list of up to 16 samples
sample_pairs = []
for cid in range(NUM_CLASSES):
    if collected[cid]:
        sample_pairs.append(collected[cid][0])
sample_pairs = sample_pairs[:16]

n_cols = 4
n_rows = (len(sample_pairs) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = axes.flatten() if isinstance(axes, np.ndarray) else [axes]

for i, (img_path, lbl_path) in enumerate(sample_pairs):
    draw_boxes(img_path, lbl_path, axes[i])
    axes[i].set_title(os.path.basename(img_path)[:30], fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "annotated_samples.png"), dpi=150)
plt.show()

# Legend
legend_patches = [mpatches.Patch(color=CLASS_COLORS[i], label=CLASS_NAMES[i])
                  for i in range(NUM_CLASSES)]
fig_leg = plt.figure(figsize=(10, 0.6))
fig_leg.legend(handles=legend_patches, loc='center', ncol=6, fontsize=9)
plt.axis('off')
plt.show()

## 6. Label Validation — Spot-Check

**Important:** The filename prefixes (e.g., `ants-1-`, `bees-103-`, `slug-101-`) are
leftover from the original Roboflow project and do **not** match the exported class IDs.
The only source of truth is the first column of each `.txt` label file.

This cell randomly samples images and prints the **raw `class_id`** from the label file
next to the class name from `data.yaml`. Visually verify that the insect inside each box
matches the displayed label. If you see an ant inside a box labeled "Bees", the dataset
label file is wrong for that image.

In [ ]:
# ============================================================
# Label Validation — Random spot-check with raw class_id info
# ============================================================
import random
random.seed(445)

print("=" * 70)
print("LABEL VALIDATION: Verify insect vs. label text")
print("=" * 70)
print("Each box shows: CLASS_NAME (raw_class_id from label file)")
print("If the visual insect does NOT match the label, the dataset is mislabeled.\n")

# Random sample from training set
all_imgs = sorted([f for f in os.listdir(TRAIN_IMG) if f.lower().endswith('.jpg')])
sample_fnames = random.sample(all_imgs, min(12, len(all_imgs)))

n_cols = 4
n_rows = 3
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 14))
axes = axes.flatten()

for idx, fname in enumerate(sample_fnames):
    img_path = os.path.join(TRAIN_IMG, fname)
    lbl_path = os.path.join(TRAIN_LBL, os.path.splitext(fname)[0] + '.txt')

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    ax = axes[idx]
    ax.imshow(img)

    label_ids = []
    if os.path.exists(lbl_path):
        with open(lbl_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(float(parts[0]))
                    cx, cy, bw, bh = map(float, parts[1:5])
                    x1 = int((cx - bw / 2) * w)
                    y1 = int((cy - bh / 2) * h)
                    box_w = int(bw * w)
                    box_h = int(bh * h)

                    color = CLASS_COLORS[cls_id]
                    rect = mpatches.Rectangle((x1, y1), box_w, box_h,
                                              fill=False, edgecolor=color, linewidth=2)
                    ax.add_patch(rect)
                    # Show name AND raw class_id for debugging
                    label_text = f"{CLASS_NAMES[cls_id]} [id={cls_id}]"
                    ax.text(x1, max(y1 - 4, 10), label_text,
                            fontsize=7, color='white', weight='bold',
                            bbox=dict(boxstyle='round,pad=0.15',
                                      facecolor=color, alpha=0.85))
                    label_ids.append(cls_id)

    short_name = fname[:45]
    if label_ids:
        id_str = ", ".join(str(i) for i in sorted(set(label_ids)))
        ax.set_title(f"{short_name}\nclass_ids in file: {id_str}",
                     fontsize=7, family='monospace')
    else:
        ax.set_title(f"{short_name}\n(no labels)", fontsize=7)

    ax.axis('off')

for j in range(len(sample_fnames), len(axes)):
    axes[j].axis('off')

plt.suptitle("Label Validation — Does the insect visually match the text on the box?",
             fontsize=13, y=0.99)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "label_validation.png"), dpi=150)
plt.show()

# ---- Filename prefix → actual class_id mapping table ----
print("\n" + "=" * 70)
print("FILENAME PREFIX → ACTUAL CLASS_ID (to show filename prefixes are misleading)")
print("=" * 70)
from collections import defaultdict
prefix_map = defaultdict(set)
all_lbls = sorted(os.listdir(TRAIN_LBL))
for fname in all_lbls:
    lbl_path = os.path.join(TRAIN_LBL, fname)
    line = open(lbl_path).readline().strip()
    if line:
        cls_id = int(float(line.split()[0]))
        # Extract Roboflow prefix (before .rf. hash)
        prefix = fname.split('.rf.')[0]  # e.g. "ants-1-_jpg"
        prefix_map[prefix].add(cls_id)

# Build a summary table
rows = []
for prefix in sorted(prefix_map.keys()):
    ids = sorted(prefix_map[prefix])
    names = ", ".join(CLASS_NAMES[i] for i in ids)
    count = sum(1 for l in all_lbls if l.startswith(prefix))
    rows.append({
        "Filename Prefix": prefix,
        "class_id(s) in label": str(ids),
        "Maps to": names,
        "# Files": count
    })

df_prefix = pd.DataFrame(rows)
print(f"\nFound {len(df_prefix)} unique filename prefixes mapping to different classes.")
print("This confirms filename prefixes are UNRELIABLE — trust the .txt label file only.\n")
display(df_prefix)

## 7. EDA Summary & Model Guidance

Based on the exploration above:

1. **Dataset size (13,143 images)** places us squarely in the 8,000–20,000 bracket, justifying
   **YOLO-m** (medium) and **RF-DETR-base** variants per the assignment size table.

2. **All images are 640×640** — the native resolution is square, so rectangular training
   (`rect=True`) is unnecessary. Starting at 640×640 input resolution is natural.

3. **Class balance** (from the distribution chart) guides the augmentation strategy: if
   imbalance exceeds 3:1, enabling mosaic/mixup in the YOLO training config will help
   up-sample rare classes.

4. **Object-size profile** (from the COCO pie chart) determines whether a higher input
   resolution (e.g., 1280) is needed. A heavy skew toward small objects makes 1280 worth
   the memory cost; a balanced profile makes 640 sufficient.

5. **Boxes-per-image density** influences the NMS `iou` threshold — dense images (>5 boxes)
   may benefit from lowering `iou` from 0.7 to 0.5–0.6 to reduce duplicate detections.

These findings are referenced in the CONFIG cells of all four model notebooks (Notebooks 2–5).

## 8. Label Conflict Resolution

We detect label conflicts by grouping images by their filename prefix (batch name) and applying a **majority vote**: if the prefix maps to class A but a few files have class B in the label file, those are likely annotation errors.

- **Scale:** 31 conflicts out of 13,140 files (0.24%)
  - Train: 27 conflicts (9 prefix groups, 3 files each)
  - Valid: 4 conflicts (4 prefix groups, 1 file each)
  - **Test: 0 conflicts** — evaluation labels are clean, no fixes needed

The cell below compares each image's label class_id against the majority class for its prefix. Setting `APPLY_FIXES = True` will overwrite the minority labels to match the prefix majority.

In [ ]:
# ============================================================
# Label Conflict Detection & Optional Fix
# ============================================================

APPLY_FIXES = False  # ← Set to True to correct conflicting labels

from collections import defaultdict, Counter
import shutil

def extract_prefix(fname):
    """Extract batch prefix before Roboflow hash."""
    prefix = fname.split('.rf.')[0]  # "ants-1-_jpg" or "ants-1-_png"
    for sep in ['-_jpg', '-_png', '-_jpeg']:
        if sep in prefix:
            prefix = prefix[:prefix.index(sep)]
            break
    return prefix

# ---- Phase 1: Audit all splits ----
print("=" * 70)
print("PHASE 1: Auditing labels across all splits")
print("=" * 70)

all_conflicts = []

for split_name, (img_dir, lbl_dir) in [
    ("train", (TRAIN_IMG, TRAIN_LBL)),
    ("valid", (VALID_IMG, VALID_LBL)),
    ("test",  (TEST_IMG, TEST_LBL)),
]:
    prefix_groups = defaultdict(list)
    for fname in sorted(os.listdir(lbl_dir)):
        if not fname.endswith('.txt'):
            continue
        fpath = os.path.join(lbl_dir, fname)
        with open(fpath, 'r') as f:
            class_ids = []
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    class_ids.append(int(float(parts[0])))
        prefix = extract_prefix(fname)
        prefix_groups[prefix].append((fname, class_ids, fpath))

    split_conflicts = []
    for prefix, entries in prefix_groups.items():
        all_ids = []
        for _, cids, _ in entries:
            all_ids.extend(cids)
        if not all_ids:
            continue
        majority_id = Counter(all_ids).most_common(1)[0][0]

        for fname, cids, fpath in entries:
            unique_ids = set(cids)
            has_majority = majority_id in unique_ids
            for cid in unique_ids:
                if cid != majority_id:
                    split_conflicts.append({
                        'split': split_name,
                        'prefix': prefix,
                        'filename': fname,
                        'label_path': fpath,
                        'wrong_id': cid,
                        'wrong_name': CLASS_NAMES[cid],
                        'correct_id': majority_id,
                        'correct_name': CLASS_NAMES[majority_id],
                        'safe_to_fix': not has_majority,  # only auto-fix if file had NO majority boxes
                    })

    all_conflicts.extend(split_conflicts)
    print(f"  {split_name:>6}: {len(split_conflicts)} conflicts")

# Display
if all_conflicts:
    df_cf = pd.DataFrame(all_conflicts)
    safe = df_cf['safe_to_fix'].sum()
    risky = len(df_cf) - safe
    print(f"\nTotal: {len(all_conflicts)} conflicts / ~13K files "
          f"({100*len(all_conflicts)/13140:.2f}%)")
    print(f"  Safe to fix (single-class files): {safe}")
    print(f"  Needs review (mixed-class files): {risky}\n")
    display(df_cf[['split', 'prefix', 'filename', 'wrong_name', 'correct_name', 'safe_to_fix']])

    # Per-class summary
    summary = Counter()
    for c in all_conflicts:
        summary[f"{c['wrong_name']} → {c['correct_name']}"] += 1
    print("\nConflict summary:")
    for k, v in summary.most_common():
        print(f"  {v:>3} files: {k}")
else:
    print("\nNo conflicts found — dataset appears clean.")

# ---- Phase 2: Apply fixes (only if APPLY_FIXES = True) ----
if APPLY_FIXES:
    print("\n" + "=" * 70)
    print("PHASE 2: Applying fixes")
    print("=" * 70)

    FIXED_DIR = os.path.join(OUTPUT_DIR, "dataset_fixed")

    if not os.path.exists(FIXED_DIR):
        print(f"Copying dataset to {FIXED_DIR} (one-time, ~537 MB)...")
        shutil.copytree(DATASET_PATH, FIXED_DIR,
                        ignore=shutil.ignore_patterns('*.py', '__pycache__'))
        print("Copy complete.")

    fixed_count = 0
    skipped_count = 0
    for c in all_conflicts:
        if not c['safe_to_fix']:
            skipped_count += 1
            continue  # skip mixed-class files — need manual review
        rel = os.path.relpath(c['label_path'], DATASET_PATH)
        fixed_path = os.path.join(FIXED_DIR, rel)
        with open(fixed_path, 'r') as f:
            lines = f.readlines()
        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) >= 5 and int(float(parts[0])) == c['wrong_id']:
                parts[0] = str(c['correct_id'])
                new_lines.append(' '.join(parts) + '\n')
            else:
                new_lines.append(line)
        with open(fixed_path, 'w') as f:
            f.writelines(new_lines)
        fixed_count += 1

    print(f"Fixed {fixed_count} labels (single-class mismatches).")
    if skipped_count:
        print(f"Skipped {skipped_count} (mixed-class files) — review manually if needed.")

    # Write corrected data.yaml
    yaml_path = os.path.join(FIXED_DIR, "data.yaml")
    yaml = f"""train: {FIXED_DIR}/train/images
val: {FIXED_DIR}/valid/images
test: {FIXED_DIR}/test/images

nc: 12
names:
  [
    "Ants",
    "Bees",
    "Beetles",
    "Caterpillars",
    "Earthworms",
    "Earwigs",
    "Grasshoppers",
    "Moths",
    "Slugs",
    "Snails",
    "Wasps",
    "Weevils",
  ]
"""
    with open(yaml_path, 'w') as f:
        f.write(yaml)

    print(f"\nCorrected data.yaml written to: {yaml_path}")
    print("Use this path in model training notebooks.")
else:
    print("\nAPPLY_FIXES = False — no changes made.")
    print("Set APPLY_FIXES = True at top of cell to fix conflicting labels.")